# Pipeline de Modelagem Complementar — Regressão e Árvore de Decisão

Complementa a clusterização (K-Means) com dois modelos supervisionados:

- **Parte A — Regressão:** modela o consumo residencial por domicílio em função do clima
  (temperatura, radiação, mês), quantificando a relação clima × demanda e produzindo as
  métricas MAE e RMSE. Compara Regressão Linear e Random Forest.
- **Parte B — Árvore de Decisão:** extrai regras interpretáveis dos segmentos comerciais
  definidos pela clusterização (renda, COP, aptidão solar), traduzindo os clusters em
  critérios acionáveis pelo time comercial.

In [ ]:
# ------------------------------------------------------------
# 1. MONTAGEM DO GOOGLE DRIVE
# ------------------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ------------------------------------------------------------
# 2. IMPORTAÇÃO DAS BIBLIOTECAS
# ------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# ------------------------------------------------------------
# 3. CARGA DOS DADOS
# ------------------------------------------------------------
PASTA = "/content/drive/MyDrive/dados_projeto_ciencia_dados"
painel  = pd.read_csv(f"{PASTA}/painel_mensal_uf.csv")     # 27 UF x 60 meses
clusters = pd.read_csv(f"{PASTA}/painel_uf_clusters.csv")  # 1 linha por UF, com segmento
print("Painel mensal:", painel.shape, "| Clusters:", clusters.shape)

## Parte A — Regressão: clima → consumo residencial

Alvo: **consumo residencial por domicílio** (MWh/domicílio/mês). Normalizar pelo número de
domicílios remove o efeito do tamanho da UF e isola a resposta do consumo ao clima.

In [ ]:
# ------------------------------------------------------------
# 4. PREPARAÇÃO — features (clima + mês + região) e alvo
# ------------------------------------------------------------
d = painel.dropna(subset=["consumo_resid_por_domicilio",
                          "temp_media_mensal", "radiacao_kwh_m2_dia"]).copy()

X = pd.get_dummies(
    d[["temp_media_mensal", "mes", "radiacao_kwh_m2_dia", "regiao_br"]],
    columns=["regiao_br", "mes"]
)
y = d["consumo_resid_por_domicilio"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print("Treino:", X_train.shape, "| Teste:", X_test.shape)

In [ ]:
# ------------------------------------------------------------
# 5. TREINO E AVALIAÇÃO — Regressão Linear vs Random Forest
# Métricas: MAE, RMSE e R²
# ------------------------------------------------------------
modelos = {
    "Regressão Linear": LinearRegression(),
    "Random Forest":    RandomForestRegressor(n_estimators=300, random_state=42),
}
resultados = []
for nome, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    resultados.append({
        "modelo": nome,
        "MAE":  mean_absolute_error(y_test, pred),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "R2":   r2_score(y_test, pred),
    })
res = pd.DataFrame(resultados).set_index("modelo").round(4)
print(res.to_string())
print("\nO Random Forest supera a Regressão Linear -> a relação clima × consumo é não linear.")
rf = modelos["Random Forest"]

In [ ]:
# ------------------------------------------------------------
# 6. A RELAÇÃO EM "U" — consumo por faixa de temperatura
# Consumo sobe no frio (aquecimento) E no calor (ar-condicionado),
# com mínimo na zona de conforto — por isso o modelo não linear vence.
# ------------------------------------------------------------
d["faixa_temp"] = pd.cut(d.temp_media_mensal, [0,18,21,24,27,40],
                         labels=["<18","18-21","21-24","24-27",">27"])
faixa = d.groupby("faixa_temp", observed=True)["consumo_resid_por_domicilio"].mean()

faixa.plot(kind="bar", figsize=(8,4), color="steelblue")
plt.title("Consumo Residencial por Domicílio x Faixa de Temperatura")
plt.xlabel("Temperatura média mensal (°C)")
plt.ylabel("Consumo médio (MWh/domicílio/mês)")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

In [ ]:
# ------------------------------------------------------------
# 7. IMPORTÂNCIA DAS VARIÁVEIS (Random Forest)
# ------------------------------------------------------------
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values().tail(8)
imp.plot(kind="barh", figsize=(8,5), color="darkorange")
plt.title("Importância das Variáveis — Random Forest")
plt.xlabel("Importância"); plt.tight_layout(); plt.show()
print("A temperatura média é o principal preditor do consumo residencial por domicílio.")

## Parte B — Árvore de Decisão: regras dos segmentos comerciais

Usa os segmentos do K-Means como alvo e extrai **regras explicáveis** (renda, COP, aptidão
solar) — transformando os clusters em critérios que o time comercial pode aplicar a uma
nova região.

In [ ]:
# ------------------------------------------------------------
# 8. TREINO DA ÁRVORE DE DECISÃO
# ------------------------------------------------------------
FEAT = ["aptidao_solar", "cop_medio_anual",
        "consumo_resid_por_domicilio", "rendimento_medio_pc"]
Xc = clusters[FEAT]
yc = clusters["segmento"]

arvore = DecisionTreeClassifier(max_depth=3, random_state=42).fit(Xc, yc)
print("Acurácia (treino):", round(arvore.score(Xc, yc), 3))
print("Acurácia (validação 5-fold):",
      round(cross_val_score(arvore, Xc, yc, cv=5).mean(), 3))
print("\nNota: com apenas 27 UFs, a árvore é usada para EXTRAIR REGRAS interpretáveis, "
      "não como modelo preditivo de produção.")

In [ ]:
# ------------------------------------------------------------
# 9. REGRAS EXTRAÍDAS (texto)
# ------------------------------------------------------------
print(export_text(arvore, feature_names=FEAT))

In [ ]:
# ------------------------------------------------------------
# 10. VISUALIZAÇÃO DA ÁRVORE
# ------------------------------------------------------------
plt.figure(figsize=(16, 8))
plot_tree(arvore, feature_names=FEAT, class_names=sorted(yc.unique()),
          filled=True, rounded=True, fontsize=9)
plt.title("Árvore de Decisão — Segmentos Comerciais por UF")
plt.tight_layout(); plt.show()

## Conclusões

- **Regressão:** a temperatura é o principal preditor do consumo residencial por domicílio;
  a relação é em "U" (consumo sobe no frio e no calor), o que faz o **Random Forest superar
  a Regressão Linear**. As métricas MAE/RMSE quantificam o erro de previsão do consumo.
- **Árvore de Decisão:** os segmentos comerciais são bem descritos por regras simples —
  a **renda** é o primeiro corte; entre os de renda alta, o **clima (COP)** separa o "Prêmio
  premium"; entre os de renda menor, a **aptidão solar** separa a "Vitrine solar". Isso dá ao
  time comercial critérios objetivos para classificar novas regiões.
- Juntos, os dois modelos complementam o K-Means: a regressão adiciona a dimensão preditiva
  (com MAE/RMSE) e a árvore adiciona a interpretabilidade dos segmentos.